"Agents as Tools" is an architectural pattern in AI systems where specialized AI agents are wrapped as callable functions (tools) that can be used by other agents. This creates a hierarchical structure where:

1. A primary "orchestrator" agent handles user interaction and determines which specialized agent to call

2. Specialized "tool agents" perform domain-specific tasks when called by the orchestrator

This approach mimics human team dynamics, where a manager coordinates specialists, each bringing unique expertise to solve complex problems. Rather than a single agent trying to handle everything, tasks are delegated to the most appropriate specialized agent.



## Key Benefits and Core Principles

The "Agents as Tools" pattern offers several advantages:

- Separation of concerns: Each agent has a focused area of responsibility, making the system easier to understand and maintain
- Hierarchical delegation: The orchestrator decides which specialist to invoke, creating a clear chain of command
- Modular architecture: Specialists can be added, removed, or modified independently without affecting the entire system
- Improved performance: Each agent can have tailored system prompts and tools optimized for its specific task


In [ ]:
import sys
print(sys.executable)

In [ ]:
!pip install -r requirements.txt

In [ ]:
import strands
import boto3
import pydantic

print("All imports successful")

In [ ]:
import os

from strands import Agent, tool
from strands_tools import file_write


In [ ]:
import json
import uuid
from datetime import datetime
from pathlib import Path

TRACE_DIR = Path(
    "/Users/rbakshi/Downloads/agentic_ai/LexiLensAI/lexilens_runtime_engine/raw_traces/strands"
)

TRACE_DIR.mkdir(parents=True, exist_ok=True)

TRACE_FILE = TRACE_DIR / "strands_runtime.jsonl"

GLOBAL_TRACE_ID = str(uuid.uuid4())


def emit_span(
    span_id,
    parent_span_id,
    agent,
    event,
    status="success",
    token_input=None,
    token_output=None,
    metadata=None
):

    span = {
        "trace_id": GLOBAL_TRACE_ID,
        "span_id": span_id,
        "parent_span_id": parent_span_id,
        "timestamp": datetime.utcnow().isoformat(),

        "agent": agent,
        "event": event,
        "status": status,

        "token_input": token_input,
        "token_output": token_output,

        "metadata": metadata or {}
    }

    with open(TRACE_FILE, "a") as f:
        f.write(json.dumps(span) + "\n")

In this module we will be creating an orchestrator based multi-agent workflow. 

<div style="text-align:left">
    <img src="images/architecture.png" width="75%" />
</div>

We will also explore `use_llm` which allows use to create nested agents.

## Research Agent

Lets first create a basic reasearch assistant with http_request tool. 

In [ ]:
RESEARCH_ASSISTANT_PROMPT = """You are a specialized research assistant. Focus only on providing
factual, well-sourced information in response to research questions.
Always cite your sources when possible."""

In [ ]:
research_agent = Agent(
    model="global.anthropic.claude-sonnet-4-5-20250929-v1:0",
    system_prompt=RESEARCH_ASSISTANT_PROMPT,
    # tools=[http_request]  # Here you can enable an agentic ai search tool
)

query = "Overview of Amazon Bedrock and its features"

# Call the agent and return its response
response = research_agent(query)

Now we can wrap this agent as a tool. Allowing other agents to interact with it. 

####  Best Practices for Agent as Tools

When implementing the "Agents as Tools" pattern with Strands Agents:

1. Clear tool documentation: Write descriptive docstrings that explain the agent's expertise
2. Focused system prompts: Keep each specialized agent tightly focused on its domain
3. Proper response handling: Use consistent patterns to extract and format responses
4. Tool selection guidance: Give the orchestrator clear criteria for when to use each specialized agen

In [ ]:
@tool
def research_assistant(query: str) -> str:

    tool_span_id = str(uuid.uuid4())

    emit_span(
        span_id=tool_span_id,
        parent_span_id="orchestrator_root",
        agent="research_assistant_tool",
        event="tool_invoked",
        metadata={
            "query": query
        }
    )

    try:

        research_agent = Agent(
            model="global.anthropic.claude-sonnet-4-5-20250929-v1:0",
            system_prompt=RESEARCH_ASSISTANT_PROMPT,
        )

        child_span_id = str(uuid.uuid4())

        emit_span(
            span_id=child_span_id,
            parent_span_id=tool_span_id,
            agent="research_agent",
            event="agent_execution_started",
            metadata={
                "query": query
            }
        )

        response = research_agent(query)

        emit_span(
            span_id=child_span_id,
            parent_span_id=tool_span_id,
            agent="research_agent",
            event="agent_execution_completed",
            metadata={
                "response_size": len(str(response))
            }
        )

        return str(response)

    except Exception as e:

        emit_span(
            span_id=tool_span_id,
            parent_span_id="orchestrator_root",
            agent="research_assistant_tool",
            event="tool_failed",
            status="failed",
            metadata={
                "error": str(e)
            }
        )

        return f"Error in research assistant: {str(e)}"

Now lets follow the best practices and create `product_recommendation_assistant`, `trip_planning_assistant`, and `orchestrator` agent.

### Product Recommendation Assistant

In [ ]:
@tool
def product_recommendation_assistant(query: str) -> str:

    tool_span_id = str(uuid.uuid4())

    emit_span(
        span_id=tool_span_id,
        parent_span_id="orchestrator_root",
        agent="product_recommendation_tool",
        event="tool_invoked",
        metadata={
            "query": query
        }
    )

    try:

        product_agent = Agent(
            model="global.anthropic.claude-sonnet-4-5-20250929-v1:0",
            system_prompt="""
            You are a specialized product recommendation assistant.
            Provide personalized product suggestions based on user preferences.
            Always cite your sources.
            """,
        )

        child_span_id = str(uuid.uuid4())

        emit_span(
            span_id=child_span_id,
            parent_span_id=tool_span_id,
            agent="product_agent",
            event="agent_execution_started",
            metadata={
                "query": query
            }
        )

        response = product_agent(query)

        emit_span(
            span_id=child_span_id,
            parent_span_id=tool_span_id,
            agent="product_agent",
            event="agent_execution_completed",
            metadata={
                "response_size": len(str(response))
            }
        )

        return str(response)

    except Exception as e:

        emit_span(
            span_id=tool_span_id,
            parent_span_id="orchestrator_root",
            agent="product_recommendation_tool",
            event="tool_failed",
            status="failed",
            metadata={
                "error": str(e)
            }
        )

        return f"Error in product recommendation: {str(e)}"

In [ ]:
product_recommendation_assistant("Product recommendations for flying cars")

### Trip Planning Assistant

In [ ]:
@tool
def trip_planning_assistant(query: str) -> str:

    tool_span_id = str(uuid.uuid4())

    emit_span(
        span_id=tool_span_id,
        parent_span_id="orchestrator_root",
        agent="trip_planning_tool",
        event="tool_invoked",
        metadata={
            "query": query
        }
    )

    try:

        travel_agent = Agent(
            model="global.anthropic.claude-sonnet-4-5-20250929-v1:0",
            system_prompt="""
            You are a specialized travel planning assistant.
            Create detailed travel itineraries based on user preferences.
            """,
        )

        child_span_id = str(uuid.uuid4())

        emit_span(
            span_id=child_span_id,
            parent_span_id=tool_span_id,
            agent="travel_agent",
            event="agent_execution_started",
            metadata={
                "query": query
            }
        )

        response = travel_agent(query)

        emit_span(
            span_id=child_span_id,
            parent_span_id=tool_span_id,
            agent="travel_agent",
            event="agent_execution_completed",
            metadata={
                "response_size": len(str(response))
            }
        )

        return str(response)

    except Exception as e:

        emit_span(
            span_id=tool_span_id,
            parent_span_id="orchestrator_root",
            agent="trip_planning_tool",
            event="tool_failed",
            status="failed",
            metadata={
                "error": str(e)
            }
        )

        return f"Error in trip planning: {str(e)}"

### Orchestrator Agent

In [ ]:
# Define orchestrator system prompt with clear tool selection guidance
MAIN_SYSTEM_PROMPT = """
You are an assistant that routes queries to specialized agents:
- For research questions and factual information → Use the research_assistant tool
- For product recommendations and shopping advice → Use the product_recommendation_assistant tool
- For travel planning and itineraries → Use the trip_planning_assistant tool
- For simple questions not requiring specialized knowledge → Answer directly

Always select the most appropriate tool based on the user's query.
"""

In [ ]:
# Strands Agents allows easy integration of agent tools
orchestrator = Agent(
    model="global.anthropic.claude-sonnet-4-5-20250929-v1:0",  # Optional: Specify the model ID
    system_prompt=MAIN_SYSTEM_PROMPT,
    tools=[
        research_assistant,
        product_recommendation_assistant,
        trip_planning_assistant,
        file_write,
    ],
)

In [ ]:
# Example: E-commerce Customer Service System
customer_query = (
    "I'm looking for hiking boots. Write the final response to current directory."
)

os.environ["BYPASS_TOOL_CONSENT"] = "true"

# The orchestrator automatically determines this requires multiple specialized agents
emit_span(
    span_id="orchestrator_root",
    parent_span_id=None,
    agent="orchestrator",
    event="workflow_started",
    metadata={
        "query": customer_query
    }
)

response = orchestrator(customer_query)

emit_span(
    span_id="orchestrator_root_end",
    parent_span_id="orchestrator_root",
    agent="orchestrator",
    event="workflow_completed"
)

Lets look at the messages of the orchestrator. Here you can see the agent decided to use the sub-agent as tool

In [ ]:
orchestrator.messages

In [ ]:
customer_query = "Can you help me plan my trip to Patagonia"

emit_span(
    span_id="orchestrator_root",
    parent_span_id=None,
    agent="orchestrator",
    event="workflow_started",
    metadata={
        "query": customer_query
    }
)

response = orchestrator(customer_query)

emit_span(
    span_id="orchestrator_root_end",
    parent_span_id="orchestrator_root",
    agent="orchestrator",
    event="workflow_completed"
)

In [ ]:
orchestrator.messages

### Calling multiple agents 

In [ ]:
orchestrator.messages = []

In [ ]:
query = "Can you do a research on spain? Also help me plan a 7 day trip."

emit_span(
    span_id="orchestrator_root",
    parent_span_id=None,
    agent="orchestrator",
    event="workflow_started",
    metadata={
        "query": query
    }
)

orchestrator(query)

emit_span(
    span_id="orchestrator_root_end",
    parent_span_id="orchestrator_root",
    agent="orchestrator",
    event="workflow_completed"
)

Behind the scenes, the orchestrator will:
1. First call the `research_assistant`
2. Then call `trip_planning_assistant`
3. Combine these specialized responses into a cohesive answer that addresses both the queries

### Sequential Agent Communication Pattern


The agent tool can also combine multiple agents together. In this example we will provide output of `research_agent` to `summary_agent` and return the summarized response.

In [ ]:
 # define the user query
topic = "generative Ai"
# Create a research agent
research_agent = Agent(
    model="global.anthropic.claude-sonnet-4-5-20250929-v1:0",
    system_prompt=RESEARCH_ASSISTANT_PROMPT,
)
# Create a summarization agent
summary_agent = Agent(
    model="global.anthropic.claude-sonnet-4-5-20250929-v1:0",
    system_prompt="""
    You are a summarization specialist focused on distilling complex information into clear, concise summaries.
    Your primary goal is to take detailed information and extract the key points, main arguments, and critical data.
    You should maintain the accuracy of the original content while making it more digestible.
    Focus on clarity, brevity, and highlighting the most important aspects of the information.
    """,
)

print("Multiple agents created successfully!")
print(f"\n🔍 RESEARCH AGENT working on: {topic}\n") 
try:
    # Agent 1: Invoke research agent
    research_response = research_agent(
        f"Please gather comprehensive information about {topic}."
    )
    research_text = research_response.message['content'][0]["text"]
    print("\n✂️ SUMMARY AGENT distilling the research\n")
    
    # Agent 2: Ask the summary agent to create a concise summary
    summary_response = summary_agent(
        f"Please create a concise summary of this research: {research_text}"
    )
    summary_text = summary_response.message['content'][0]["text"]
    
    print(summary_text)
except Exception as e:
    print(f"Error in research assistant: {str(e)}")

## Congrats!

You've learned how to use agents as tools in Strands Agents to create more complex agentic applications

---# Part 2: From Manual Instrumentation to Auto-Observability## The Problem: Manual Tracing Doesn't ScaleLook at what we just did above. Every single tool-agent required **~50 lines of boilerplate** just to emit basic spans. Let's quantify it:

In [ ]:
# Let's count the observability tax in our code above

# Each tool function has ~20 lines of emit_span() calls
# For 3 tools + orchestrator calls, that's:
manual_lines_per_tool = 20  # emit_span wrappers, try/except, span_id generation
num_tools = 3
orchestrator_overhead = 10  # start/end spans around each orchestrator call

total_manual_lines = (manual_lines_per_tool * num_tools) + orchestrator_overhead
agent_logic_lines = 5 * num_tools  # actual agent creation + invocation per tool

print("=" * 60)
print("OBSERVABILITY TAX ANALYSIS")
print("=" * 60)
print(f"")
print(f"Manual tracing lines per tool:     {manual_lines_per_tool}")
print(f"Number of tools:                   {num_tools}")
print(f"Orchestrator overhead lines:       {orchestrator_overhead}")
print(f"")
print(f"Total manual observability code:   {total_manual_lines} lines")
print(f"Total actual agent logic:          {agent_logic_lines} lines")
print(f"")
print(f"Ratio: {total_manual_lines/agent_logic_lines:.1f}x more observability code than agent code!")
print(f"")
print("Problems with this approach:")
print("  1. Fragile: forget one emit_span() and you lose visibility")
print("  2. No session awareness: spans are flat, no execution lineage")
print("  3. No anomaly detection: all spans show 'success' even on silent failures")
print("  4. Scales linearly: 10 tools = 200+ lines of boilerplate")
print("  5. No context tracking: can't see what information flows between agents")
print("=" * 60)

## Solution: LexiLensAI Auto-InstrumentationWhat if you could get full session-native observability with **one line of code**?Below is the `lexilens` SDK — it monkey-patches the Strands Agent class to automatically emit OpenTelemetry-compatible spans for every agent invocation, tool call, and delegation. Zero changes to your agent code.

In [ ]:
# lexilens/sdk.py - Session-native auto-instrumentation for AI agents
# This is a working stub of the LexiLensAI SDK

import json
import uuid
import time
import functools
from datetime import datetime, timezone
from pathlib import Path
from dataclasses import dataclass, field, asdict
from typing import Optional, Dict, List, Any
from collections import defaultdict


@dataclass
class Span:
    trace_id: str
    span_id: str
    parent_span_id: Optional[str]
    session_id: str
    agent_name: str
    event: str
    status: str = "success"
    start_time: Optional[str] = None
    end_time: Optional[str] = None
    duration_ms: Optional[float] = None
    token_input: Optional[int] = None
    token_output: Optional[int] = None
    metadata: Dict = field(default_factory=dict)


class SessionTracker:
    """Tracks execution at the SESSION level, not just span level."""
    
    def __init__(self, session_id: str):
        self.session_id = session_id
        self.spans: List[Span] = []
        self.delegation_chain: List[str] = []
        self.total_tokens_in = 0
        self.total_tokens_out = 0
        self.tool_call_counts: Dict[str, int] = defaultdict(int)
        self.retry_tracker: Dict[str, List[float]] = defaultdict(list)
        self.anomalies: List[Dict] = []
    
    def add_span(self, span: Span):
        self.spans.append(span)
        if span.token_input:
            self.total_tokens_in += span.token_input
        if span.token_output:
            self.total_tokens_out += span.token_output
        
        # Track tool calls for retry detection
        tool_key = f"{span.agent_name}:{span.metadata.get('input', '')[:50]}"
        self.tool_call_counts[span.agent_name] += 1
        self.retry_tracker[tool_key].append(time.time())
        
        # Anomaly detection: retry storm (same agent + same input 3x in 60s)
        self._check_retry_storm(tool_key)
        
        # Anomaly detection: token explosion
        self._check_token_explosion(span)
    
    def _check_retry_storm(self, tool_key: str):
        calls = self.retry_tracker[tool_key]
        if len(calls) >= 3:
            recent = [t for t in calls if time.time() - t < 60]
            if len(recent) >= 3:
                self.anomalies.append({
                    "type": "retry_storm",
                    "severity": "high",
                    "agent": tool_key.split(":")[0],
                    "detail": f"Agent called {len(recent)}x in <60s with similar input",
                    "timestamp": datetime.now(timezone.utc).isoformat()
                })
    
    def _check_token_explosion(self, span: Span):
        if span.token_output and span.token_output > 4000:
            self.anomalies.append({
                "type": "token_explosion",
                "severity": "medium",
                "agent": span.agent_name,
                "detail": f"Single response used {span.token_output} output tokens",
                "timestamp": datetime.now(timezone.utc).isoformat()
            })
    
    def get_delegation_graph(self) -> Dict[str, List[str]]:
        """Returns adjacency list of agent delegations."""
        graph = defaultdict(set)
        span_map = {s.span_id: s for s in self.spans}
        for span in self.spans:
            if span.parent_span_id and span.parent_span_id in span_map:
                parent = span_map[span.parent_span_id]
                if parent.agent_name != span.agent_name:
                    graph[parent.agent_name].add(span.agent_name)
        return {k: list(v) for k, v in graph.items()}
    
    def summary(self) -> Dict:
        return {
            "session_id": self.session_id,
            "total_spans": len(self.spans),
            "total_tokens": {"input": self.total_tokens_in, "output": self.total_tokens_out},
            "unique_agents": list(set(s.agent_name for s in self.spans)),
            "delegation_graph": self.get_delegation_graph(),
            "anomalies": self.anomalies,
            "duration_ms": self._session_duration()
        }
    
    def _session_duration(self) -> Optional[float]:
        if len(self.spans) < 2:
            return None
        first = self.spans[0].start_time
        last = self.spans[-1].end_time or self.spans[-1].start_time
        if first and last:
            t1 = datetime.fromisoformat(first)
            t2 = datetime.fromisoformat(last)
            return (t2 - t1).total_seconds() * 1000
        return None


def _extract_agent_name(agent_self) -> str:
    """Extract a meaningful name from a Strands Agent instance."""
    # 1. Check for explicit name attribute
    name = getattr(agent_self, 'name', None)
    if name and name not in ("Agent", "Strands Agents"):
        return name
    
    # 2. Use system prompt to derive a descriptive name
    prompt = getattr(agent_self, 'system_prompt', None)
    if prompt:
        prompt_clean = prompt.strip()
        # Extract role from common patterns like "You are a specialized X assistant"
        lower = prompt_clean.lower()
        if "research" in lower:
            return "research_agent"
        elif "product" in lower or "recommendation" in lower:
            return "product_recommendation_agent"
        elif "travel" in lower or "trip" in lower or "itinerar" in lower:
            return "trip_planning_agent"
        elif "summar" in lower:
            return "summary_agent"
        elif "route" in lower or "orchestrat" in lower or "specialized agents" in lower:
            return "orchestrator"
        else:
            # Use first 40 chars of prompt as name, cleaned up
            snippet = prompt_clean[:40].replace("\n", " ").strip()
            return snippet.replace(" ", "_")[:30]
    
    return "unnamed_agent"


def _extract_tokens_from_response(response) -> tuple:
    """Extract token counts from Strands Agent response object."""
    token_in = None
    token_out = None
    
    # Method 1: response.message.metadata.usage (Strands pattern)
    try:
        msg = getattr(response, 'message', None)
        if msg and isinstance(msg, dict):
            metadata = msg.get('metadata', {})
            usage = metadata.get('usage', {})
            token_in = usage.get('inputTokens') or usage.get('input_tokens')
            token_out = usage.get('outputTokens') or usage.get('output_tokens')
            if token_in:
                return (token_in, token_out)
    except (AttributeError, TypeError, KeyError):
        pass
    
    # Method 2: response.metrics
    try:
        metrics = getattr(response, 'metrics', None)
        if metrics:
            token_in = getattr(metrics, 'inputTokens', None) or getattr(metrics, 'input_tokens', None)
            token_out = getattr(metrics, 'outputTokens', None) or getattr(metrics, 'output_tokens', None)
            if token_in:
                return (token_in, token_out)
    except (AttributeError, TypeError):
        pass
    
    # Method 3: response.usage dict
    try:
        usage = getattr(response, 'usage', None)
        if usage and isinstance(usage, dict):
            token_in = usage.get('inputTokens') or usage.get('input_tokens')
            token_out = usage.get('outputTokens') or usage.get('output_tokens')
            if token_in:
                return (token_in, token_out)
    except (AttributeError, TypeError):
        pass
    
    return (token_in, token_out)


class LexiLens:
    """
    LexiLensAI Auto-Instrumentation SDK.
    
    Usage:
        lexilens = LexiLens.init(service_name="my-agent-app")
        # That's it. All Strands Agent calls are now traced.
    """
    
    _instance = None
    
    def __init__(self, service_name: str, output_dir: Optional[str] = None):
        self.service_name = service_name
        self.trace_id = str(uuid.uuid4())
        self.session_id = str(uuid.uuid4())
        self.output_dir = Path(output_dir) if output_dir else Path("./lexilens_traces")
        self.output_dir.mkdir(parents=True, exist_ok=True)
        self.trace_file = self.output_dir / f"session_{self.session_id[:8]}.jsonl"
        self.session = SessionTracker(self.session_id)
        self._span_stack: List[str] = []
        self._original_agent_call = None
        self._patched = False
    
    @classmethod
    def init(cls, service_name: str = "strands-app", output_dir: Optional[str] = None) -> 'LexiLens':
        """Initialize LexiLensAI. Patches Strands Agent automatically."""
        instance = cls(service_name=service_name, output_dir=output_dir)
        instance._patch_strands()
        cls._instance = instance
        print(f"[LexiLens] Initialized for '{service_name}'")
        print(f"[LexiLens] Session: {instance.session_id[:8]}...")
        print(f"[LexiLens] Traces -> {instance.trace_file}")
        return instance
    
    def _patch_strands(self):
        """Monkey-patch strands.Agent.__call__ to auto-emit spans."""
        if self._patched:
            return
        
        from strands import Agent as StrandsAgent
        original_call = StrandsAgent.__call__
        lexilens_ref = self
        
        @functools.wraps(original_call)
        def instrumented_call(agent_self, *args, **kwargs):
            agent_name = _extract_agent_name(agent_self)
            
            # Determine parent from call stack
            parent_span_id = lexilens_ref._span_stack[-1] if lexilens_ref._span_stack else None
            span_id = str(uuid.uuid4())
            
            # Push onto stack for child detection
            lexilens_ref._span_stack.append(span_id)
            
            start = datetime.now(timezone.utc)
            span = Span(
                trace_id=lexilens_ref.trace_id,
                span_id=span_id,
                parent_span_id=parent_span_id,
                session_id=lexilens_ref.session_id,
                agent_name=agent_name,
                event="agent_invocation",
                start_time=start.isoformat(),
                metadata={"input": str(args[0])[:200] if args else ""}
            )
            
            try:
                response = original_call(agent_self, *args, **kwargs)
                
                end = datetime.now(timezone.utc)
                span.end_time = end.isoformat()
                span.duration_ms = (end - start).total_seconds() * 1000
                span.status = "success"
                
                # Extract token usage from Strands response
                token_in, token_out = _extract_tokens_from_response(response)
                span.token_input = token_in
                span.token_output = token_out
                
                span.metadata["output_size"] = len(str(response))
                
                return response
                
            except Exception as e:
                end = datetime.now(timezone.utc)
                span.end_time = end.isoformat()
                span.duration_ms = (end - start).total_seconds() * 1000
                span.status = "error"
                span.metadata["error"] = str(e)
                raise
                
            finally:
                # Record span
                lexilens_ref.session.add_span(span)
                lexilens_ref._write_span(span)
                lexilens_ref._span_stack.pop()
        
        StrandsAgent.__call__ = instrumented_call
        self._original_agent_call = original_call
        self._patched = True
    
    def _write_span(self, span: Span):
        with open(self.trace_file, "a") as f:
            f.write(json.dumps(asdict(span)) + "\n")
    
    def report(self):
        """Print session summary with anomaly detection results."""
        s = self.session.summary()
        print("\n" + "=" * 60)
        print("LEXILENS SESSION REPORT")
        print("=" * 60)
        print(f"Session:        {s['session_id'][:8]}...")
        print(f"Total spans:    {s['total_spans']}")
        print(f"Tokens:         {s['total_tokens']['input'] or 0} in / {s['total_tokens']['output'] or 0} out")
        print(f"Agents used:    {', '.join(s['unique_agents'])}")
        if s['duration_ms']:
            print(f"Duration:       {s['duration_ms']:.0f}ms")
        else:
            print("Duration:       N/A")
        
        if s['delegation_graph']:
            print(f"\nDelegation Graph:")
            for parent, children in s['delegation_graph'].items():
                for child in children:
                    print(f"  {parent} -> {child}")
        
        if s['anomalies']:
            print(f"\n  ANOMALIES DETECTED: {len(s['anomalies'])}")
            for a in s['anomalies']:
                print(f"  [{a['severity'].upper()}] {a['type']}: {a['detail']}")
        else:
            print(f"\n  No anomalies detected")
        
        print("=" * 60)
        return s
    
    def unpatch(self):
        """Restore original Strands Agent behavior."""
        if self._original_agent_call and self._patched:
            from strands import Agent as StrandsAgent
            StrandsAgent.__call__ = self._original_agent_call
            self._patched = False
            print("[LexiLens] Unpatched - original Agent behavior restored")


print("LexiLens SDK loaded")

## The "After": Same Agents, Zero Manual SpansNow let's run the **exact same orchestrator pattern** — but with LexiLensAI auto-instrumentation instead of manual `emit_span()` calls.Notice: **no changes to agent code.** One `init()` call does everything.

In [ ]:
# Initialize LexiLensAI - this is the ONLY observability code you need
lexilens = LexiLens.init(
    service_name="strands-multi-agent-demo",
    output_dir="/Users/rbakshi/Downloads/agentic_ai/LexiLensAI/lexilens_runtime_engine/raw_traces/strands_auto"
)

In [ ]:
# LOOK: No emit_span() anywhere. Same agent logic, zero observability tax.

@tool
def research_assistant_v2(query: str) -> str:
    """Research specialist - provides factual, well-sourced information."""
    agent = Agent(
        model="global.anthropic.claude-sonnet-4-5-20250929-v1:0",
        system_prompt=RESEARCH_ASSISTANT_PROMPT,
    )
    response = agent(query)
    return str(response)


@tool  
def product_recommendation_v2(query: str) -> str:
    """Product recommendation specialist - personalized suggestions."""
    agent = Agent(
        model="global.anthropic.claude-sonnet-4-5-20250929-v1:0",
        system_prompt="You are a specialized product recommendation assistant. Provide personalized product suggestions.",
    )
    response = agent(query)
    return str(response)


@tool
def trip_planning_v2(query: str) -> str:
    """Travel planning specialist - detailed itineraries."""
    agent = Agent(
        model="global.anthropic.claude-sonnet-4-5-20250929-v1:0",
        system_prompt="You are a specialized travel planning assistant. Create detailed travel itineraries.",
    )
    response = agent(query)
    return str(response)


# Compare the line counts:
print("BEFORE (with manual emit_span):")
print("  research_assistant:    65 lines")
print("  product_recommendation: 65 lines")  
print("  trip_planning:          65 lines")
print("  Total:                 195 lines\n")
print("AFTER (with LexiLensAI):")
print("  research_assistant_v2:  6 lines")
print("  product_recommendation_v2: 6 lines")
print("  trip_planning_v2:       6 lines")
print("  Total:                  18 lines")
print(f"\n  -> {195//18}x reduction in code, BETTER observability")

In [ ]:
# Create orchestrator with clean tools
orchestrator_v2 = Agent(
    model="global.anthropic.claude-sonnet-4-5-20250929-v1:0",
    system_prompt=MAIN_SYSTEM_PROMPT,
    tools=[research_assistant_v2, product_recommendation_v2, trip_planning_v2],
)

os.environ["BYPASS_TOOL_CONSENT"] = "true"

# Run a multi-agent query - LexiLens captures everything automatically
response = orchestrator_v2("Research Spain and plan me a 7-day trip with hotel recommendations")

In [ ]:
# One call to see the full session picture
session_data = lexilens.report()

## Session Graph VisualizationLexiLensAI reconstructs the execution as a **directed session graph** — showing exactly which agent delegated to which, the token cost at each node, and where anomalies occurred.

In [ ]:
# Install matplotlib into notebook venv and import it
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "matplotlib", "-q"],
                      stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
print(f"matplotlib {matplotlib.__version__} loaded from {matplotlib.__file__}")


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from collections import defaultdict

def visualize_session_graph(session, title="Agent Session Graph"):
    """
    Renders the session delegation graph using matplotlib.
    Nodes = agents, Edges = delegations, Node size = token usage.
    """
    
    spans = session.spans
    if not spans:
        print("No spans recorded yet.")
        return
    
    # Build graph data
    nodes = {}
    edges = []
    
    span_map = {s.span_id: s for s in spans}
    
    for span in spans:
        name = span.agent_name
        if name not in nodes:
            nodes[name] = {"tokens_in": 0, "tokens_out": 0, "calls": 0, "has_anomaly": False}
        nodes[name]["tokens_in"] += span.token_input or 0
        nodes[name]["tokens_out"] += span.token_output or 0
        nodes[name]["calls"] += 1
        
        if span.parent_span_id and span.parent_span_id in span_map:
            parent_name = span_map[span.parent_span_id].agent_name
            edges.append((parent_name, name))
    
    # Check anomalies
    for anomaly in session.anomalies:
        agent = anomaly.get("agent", "")
        if agent in nodes:
            nodes[agent]["has_anomaly"] = True
    
    # Layout: simple hierarchical
    levels = defaultdict(list)
    all_targets = set(e[1] for e in edges)
    all_sources = set(e[0] for e in edges)
    roots = all_sources - all_targets
    
    if not roots:
        roots = {spans[0].agent_name}
    
    # BFS for levels
    visited = set()
    queue = [(r, 0) for r in roots]
    while queue:
        node, level = queue.pop(0)
        if node in visited:
            continue
        visited.add(node)
        levels[level].append(node)
        for src, tgt in edges:
            if src == node and tgt not in visited:
                queue.append((tgt, level + 1))
    
    # Add unvisited nodes
    for name in nodes:
        if name not in visited:
            max_level = max(levels.keys()) + 1 if levels else 0
            levels[max_level].append(name)
    
    # Calculate positions
    positions = {}
    for level, names in levels.items():
        for i, name in enumerate(names):
            x = (i + 0.5) * (10.0 / max(len(names), 1))
            y = -level * 2.5
            positions[name] = (x, y)
    
    # Draw
    fig, ax = plt.subplots(1, 1, figsize=(12, 8))
    ax.set_xlim(-1, 11)
    y_vals = [p[1] for p in positions.values()]
    ax.set_ylim(min(y_vals) - 2, max(y_vals) + 2)
    ax.axis('off')
    ax.set_title(title, fontsize=16, fontweight='bold', pad=20)
    
    # Draw edges
    drawn_edges = set()
    for src, tgt in edges:
        edge_key = (src, tgt)
        if edge_key in drawn_edges:
            continue
        drawn_edges.add(edge_key)
        
        if src in positions and tgt in positions:
            x1, y1 = positions[src]
            x2, y2 = positions[tgt]
            ax.annotate("",
                xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle="->", color="#555555", lw=2))
    
    # Draw nodes
    for name, pos in positions.items():
        info = nodes.get(name, {"tokens_in": 0, "tokens_out": 0, "calls": 0, "has_anomaly": False})
        
        if info["has_anomaly"]:
            color = "#FF4444"
            edge_color = "#CC0000"
        elif name in roots:
            color = "#4A90D9"
            edge_color = "#2A5F8F"
        else:
            color = "#5CB85C"
            edge_color = "#3D8B3D"
        
        circle = plt.Circle(pos, 0.6, color=color, ec=edge_color, lw=2, zorder=5)
        ax.add_patch(circle)
        
        short_name = name[:20] if len(name) > 20 else name
        ax.text(pos[0], pos[1] + 0.05, short_name, ha='center', va='center',
                fontsize=8, fontweight='bold', color='white', zorder=6)
        
        token_str = f"{info['tokens_in']+info['tokens_out']} tok | {info['calls']}x"
        ax.text(pos[0], pos[1] - 0.9, token_str, ha='center', va='top',
                fontsize=7, color='#666666')
    
    # Legend
    legend_elements = [
        mpatches.Patch(color='#4A90D9', label='Orchestrator'),
        mpatches.Patch(color='#5CB85C', label='Specialist Agent'),
        mpatches.Patch(color='#FF4444', label='Anomaly Detected'),
    ]
    ax.legend(handles=legend_elements, loc='lower right', fontsize=9)
    
    plt.tight_layout()
    plt.savefig("session_graph.png", dpi=150, bbox_inches='tight')
    plt.show()
    print(f"\nGraph saved to session_graph.png")


# Render the session graph from our auto-instrumented run
visualize_session_graph(lexilens.session, title="Multi-Agent Session Graph - Auto-Instrumented by LexiLensAI")

## Anomaly Detection: Catching Silent FailuresTraditional tracing shows "success" for every span. But LexiLensAI detects **session-level anomalies** — problems that only become visible when you look at the execution as a whole.Let's simulate a retry storm (agent calling itself repeatedly with the same query):

In [ ]:
# Simulate a retry storm - agent called 5x with same input in rapid succession
# In production, this happens when an agent isn't satisfied with its own output
# and re-queries. Each individual call "succeeds" but the session is burning tokens.

import time

print("Simulating retry storm...\n")

for i in range(5):
    fake_span = Span(
        trace_id=lexilens.trace_id,
        span_id=str(uuid.uuid4()),
        parent_span_id=lexilens._span_stack[-1] if lexilens._span_stack else None,
        session_id=lexilens.session_id,
        agent_name="research_agent_retry_sim",
        event="agent_invocation",
        status="success",  # Each call "succeeds"!
        start_time=datetime.now(timezone.utc).isoformat(),
        end_time=datetime.now(timezone.utc).isoformat(),
        duration_ms=1200 + (i * 300),
        token_input=500,
        token_output=2000,
        metadata={"query": "What are the best hotels in Barcelona?"}  # Same query each time
    )
    lexilens.session.add_span(fake_span)
    time.sleep(0.1)  # Rapid succession

# Now check what LexiLens caught
print("All 5 calls returned 'success'. Traditional tracing sees nothing wrong.")
print("But LexiLensAI detects:\n")

lexilens.report()

## Key Insight| Aspect | Manual emit_span() | LexiLensAI ||--------|-------------------|------------|| Code overhead | 50+ lines per tool | 1 line total || Session awareness | None (flat spans) | Full execution graph || Anomaly detection | None | Retry storms, token explosion, context drift || Delegation tracking | Manual parent_span_id | Automatic via call stack || Works across frameworks | Custom per framework | Strands, Claude SDK, LangChain |**The difference isn't just less code. It's a fundamentally different unit of analysis: the Session, not the Span.**---*Built by CurvSort | [github.com/cuvsort/agent-session-graph](https://github.com/cuvsort/agent-session-graph) | [curvsort.com/lexilensai](https://www.curvsort.com/lexilensai)*